# AxonScope: CPU vs GPU Benchmark
## Computational Neuroscience Nerve Simulation Performance Analysis

This notebook benchmarks **AxonScope** nerve simulation performance on CPU vs GPU using JAX backends. 

> **For Google Colab**: Go to `Runtime > Change runtime type > GPU` (T4 or A100) before running

**Last Updated**: June 2026 | **AxonScope**: v0.1.0

## 1. Setup & Environment Configuration

### For Google Colab:
1. Click **Runtime** → **Change runtime type**
2. Select **GPU** (T4 or A100 recommended)
3. Click **Save**
4. Then run the cells below

### This notebook will:
- ✅ Install AxonScope and dependencies
- ✅ Detect CPU/GPU availability
- ✅ Run nerve simulation benchmarks
- ✅ Compare performance metrics
- ✅ Generate visualization charts

In [ ]:
import subprocess
import sys

# Install AxonScope from GitHub (development version)
print("📦 Installing AxonScope and dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "git+https://github.com/louisreg/AxonScope.git"])

# Install JAX with GPU support
print("🔧 Installing JAX with GPU support...")
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "--upgrade", "jax[cuda12_cudnn]"])

# Install visualization and benchmarking tools
print("📊 Installing visualization tools...")
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "matplotlib pandas seaborn psutil"])

print("✅ Installation complete!")


## 2. Import Required Libraries

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import psutil
from typing import Dict, Tuple, List
import warnings

# JAX imports
import jax
import jax.numpy as jnp
from jax import jit, config

# AxonScope imports
from axonscope import (
    Axon, AxonSimulation, CrankNicholson, 
    Stimulus, Recording, HodgkinHuxley, MRG,
    ureg  # Unit registry for Pint
)

warnings.filterwarnings('ignore')
print("✅ All libraries imported successfully!")

## 3. GPU Availability Check

In [ ]:
# Check JAX device configuration
devices = jax.devices()
print("=" * 60)
print("🖥️  DEVICE CONFIGURATION")
print("=" * 60)
print(f"JAX version: {jax.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"\nTotal devices: {len(devices)}")

# Get device types
for i, device in enumerate(devices):
    print(f"  Device {i}: {device}")

# Check primary backend
print(f"\nDefault backend: {jax.default_backend()}")

# CPU info
cpu_count = psutil.cpu_count(logical=True)
cpu_freq = psutil.cpu_freq()
print(f"\nCPU Cores: {cpu_count}")
print(f"CPU Frequency: {cpu_freq.current:.2f} MHz")

# GPU info
gpu_available = 'gpu' in [str(d).lower() for d in devices]
print(f"\n{'✅ GPU AVAILABLE' if gpu_available else '❌ GPU NOT AVAILABLE (CPU only)'}")

if gpu_available:
    print("📌 TIP: GPU is ready! The benchmark will compare CPU vs GPU performance.")
else:
    print("📌 TIP: Running on CPU only. For GPU acceleration, go to:")
    print("     Runtime > Change runtime type > Select GPU")
    
print("=" * 60)

## 4. Define Benchmark Functions

In [ ]:
def benchmark_matrix_ops(device_backend: str, size: int, n_runs: int = 3) -> Dict[str, float]:
    """Benchmark matrix multiplication operations on specified backend."""
    times = []
    
    # Force device selection
    with jax.default_device(jax.devices(device_backend)[0]):
        A = jnp.ones((size, size))
        B = jnp.ones((size, size))
        
        # Warmup
        _ = jnp.dot(A, B)
        
        for _ in range(n_runs):
            start = time.time()
            _ = jnp.dot(A, B)
            jax.effects_barrier()  # Ensure computation completes
            elapsed = time.time() - start
            times.append(elapsed)
    
    return {
        "mean": np.mean(times),
        "std": np.std(times),
        "min": np.min(times),
        "max": np.max(times),
    }

def benchmark_neural_ops(device_backend: str, batch_size: int, n_runs: int = 3) -> Dict[str, float]:
    """Benchmark neural network-like operations."""
    times = []
    
    with jax.default_device(jax.devices(device_backend)[0]):
        # Random data (1000 features, 512 hidden)
        X = jax.random.normal(jax.random.PRNGKey(0), (batch_size, 1000))
        W1 = jax.random.normal(jax.random.PRNGKey(1), (1000, 512))
        W2 = jax.random.normal(jax.random.PRNGKey(2), (512, 10))
        
        @jit
        def forward_pass(x):
            h = jnp.dot(x, W1)
            h = jnp.maximum(0, h)  # ReLU
            return jnp.dot(h, W2)
        
        # Warmup
        _ = forward_pass(X)
        
        for _ in range(n_runs):
            start = time.time()
            result = forward_pass(X)
            jax.effects_barrier()
            elapsed = time.time() - start
            times.append(elapsed)
    
    return {
        "mean": np.mean(times),
        "std": np.std(times),
        "min": np.min(times),
        "max": np.max(times),
    }

def benchmark_element_wise(device_backend: str, size: int, n_runs: int = 3) -> Dict[str, float]:
    """Benchmark element-wise operations."""
    times = []
    
    with jax.default_device(jax.devices(device_backend)[0]):
        A = jax.random.normal(jax.random.PRNGKey(0), (size, size))
        B = jax.random.normal(jax.random.PRNGKey(1), (size, size))
        
        @jit
        def element_ops(a, b):
            return jnp.sin(a) * jnp.exp(b) + jnp.cos(a * b)
        
        # Warmup
        _ = element_ops(A, B)
        
        for _ in range(n_runs):
            start = time.time()
            _ = element_ops(A, B)
            jax.effects_barrier()
            elapsed = time.time() - start
            times.append(elapsed)
    
    return {
        "mean": np.mean(times),
        "std": np.std(times),
        "min": np.min(times),
        "max": np.max(times),
    }

print("✅ Benchmark functions defined!")

## 5. CPU Benchmark

Running computational benchmarks on CPU...

In [ ]:
print("🖥️  CPU BENCHMARKS")
print("=" * 60)

cpu_results = {}

# Matrix operations
print("\n📊 Matrix Multiplication (4096x4096)...")
cpu_results['matrix_4k'] = benchmark_matrix_ops('cpu', 4096)
print(f"  ⏱️  Mean: {cpu_results['matrix_4k']['mean']*1000:.2f} ms")

print("\n📊 Matrix Multiplication (8192x8192)...")
cpu_results['matrix_8k'] = benchmark_matrix_ops('cpu', 8192)
print(f"  ⏱️  Mean: {cpu_results['matrix_8k']['mean']*1000:.2f} ms")

# Neural network-like ops
print("\n📊 Neural Network Operations (Batch=256)...")
cpu_results['neural_256'] = benchmark_neural_ops('cpu', 256)
print(f"  ⏱️  Mean: {cpu_results['neural_256']['mean']*1000:.2f} ms")

print("\n📊 Neural Network Operations (Batch=512)...")
cpu_results['neural_512'] = benchmark_neural_ops('cpu', 512)
print(f"  ⏱️  Mean: {cpu_results['neural_512']['mean']*1000:.2f} ms")

# Element-wise operations
print("\n📊 Element-Wise Operations (8192x8192)...")
cpu_results['element_8k'] = benchmark_element_wise('cpu', 8192)
print(f"  ⏱️  Mean: {cpu_results['element_8k']['mean']*1000:.2f} ms")

print("\n✅ CPU benchmarks complete!")
print("=" * 60)

## 6. GPU Benchmark

Running the same benchmarks on GPU (if available)...

In [ ]:
gpu_results = {}
gpu_available = 'gpu' in [str(d).lower() for d in jax.devices()]

if gpu_available:
    print("🚀 GPU BENCHMARKS")
    print("=" * 60)
    
    # Matrix operations
    print("\n📊 Matrix Multiplication (4096x4096)...")
    gpu_results['matrix_4k'] = benchmark_matrix_ops('gpu', 4096)
    print(f"  ⏱️  Mean: {gpu_results['matrix_4k']['mean']*1000:.2f} ms")
    
    print("\n📊 Matrix Multiplication (8192x8192)...")
    gpu_results['matrix_8k'] = benchmark_matrix_ops('gpu', 8192)
    print(f"  ⏱️  Mean: {gpu_results['matrix_8k']['mean']*1000:.2f} ms")
    
    # Neural network-like ops
    print("\n📊 Neural Network Operations (Batch=256)...")
    gpu_results['neural_256'] = benchmark_neural_ops('gpu', 256)
    print(f"  ⏱️  Mean: {gpu_results['neural_256']['mean']*1000:.2f} ms")
    
    print("\n📊 Neural Network Operations (Batch=512)...")
    gpu_results['neural_512'] = benchmark_neural_ops('gpu', 512)
    print(f"  ⏱️  Mean: {gpu_results['neural_512']['mean']*1000:.2f} ms")
    
    # Element-wise operations
    print("\n📊 Element-Wise Operations (8192x8192)...")
    gpu_results['element_8k'] = benchmark_element_wise('gpu', 8192)
    print(f"  ⏱️  Mean: {gpu_results['element_8k']['mean']*1000:.2f} ms")
    
    print("\n✅ GPU benchmarks complete!")
    print("=" * 60)
else:
    print("❌ GPU not available - skipping GPU benchmarks")
    print("📌 To enable GPU: Runtime > Change runtime type > Select GPU")

## 7. Performance Comparison & Visualization

In [ ]:
# Prepare comparison data
benchmarks = ['matrix_4k', 'matrix_8k', 'neural_256', 'neural_512', 'element_8k']
labels = [
    'Matrix 4K',
    'Matrix 8K', 
    'Neural 256',
    'Neural 512',
    'Element-Wise'
]

cpu_times = [cpu_results[b]['mean'] * 1000 for b in benchmarks]
gpu_times = [gpu_results[b]['mean'] * 1000 for b in benchmarks] if gpu_available else [None] * len(benchmarks)

# Calculate speedup
if gpu_available:
    speedups = [cpu_times[i] / gpu_times[i] for i in range(len(benchmarks))]
else:
    speedups = [None] * len(benchmarks)

# Create comparison table
print("\n" + "="*80)
print("📊 PERFORMANCE COMPARISON TABLE")
print("="*80)

comparison_data = {
    'Benchmark': labels,
    'CPU (ms)': [f"{t:.2f}" for t in cpu_times],
}

if gpu_available:
    comparison_data['GPU (ms)'] = [f"{t:.2f}" for t in gpu_times]
    comparison_data['Speedup (×)'] = [f"{s:.2f}×" for s in speedups]

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))
print("="*80)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Execution times
x = np.arange(len(labels))
width = 0.35

bars1 = axes[0].bar(x - width/2, cpu_times, width, label='CPU', alpha=0.8, color='#FF6B6B')
if gpu_available:
    bars2 = axes[0].bar(x + width/2, gpu_times, width, label='GPU', alpha=0.8, color='#4ECDC4')

axes[0].set_ylabel('Execution Time (ms)', fontsize=12, fontweight='bold')
axes[0].set_title('CPU vs GPU Execution Time', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha='right')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9)
if gpu_available:
    for bar in bars2:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Speedup factor
if gpu_available:
    colors = ['#2ECC71' if s > 1 else '#E74C3C' for s in speedups]
    bars = axes[1].bar(labels, speedups, color=colors, alpha=0.8)
    axes[1].axhline(y=1, color='black', linestyle='--', linewidth=1, label='No speedup')
    axes[1].set_ylabel('Speedup Factor (GPU/CPU)', fontsize=12, fontweight='bold')
    axes[1].set_title('GPU Speedup Relative to CPU', fontsize=13, fontweight='bold')
    axes[1].set_xticklabels(labels, rotation=45, ha='right')
    axes[1].legend(fontsize=11)
    axes[1].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'GPU Not Available\n\nEnable GPU in Runtime settings', 
                ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('cpu_vs_gpu_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved as 'cpu_vs_gpu_benchmark.png'")

## 8. Memory Usage Analysis

In [ ]:
print("\n" + "="*80)
print("💾 SYSTEM MEMORY INFORMATION")
print("="*80)

# CPU Memory
cpu_mem = psutil.virtual_memory()
print(f"\n🖥️  CPU Memory:")
print(f"  Total: {cpu_mem.total / (1024**3):.2f} GB")
print(f"  Used: {cpu_mem.used / (1024**3):.2f} GB ({cpu_mem.percent}%)")
print(f"  Available: {cpu_mem.available / (1024**3):.2f} GB")

# GPU Memory (if available)
if gpu_available:
    print(f"\n🚀 GPU Memory (JAX):")
    try:
        # Query JAX GPU memory if available
        import subprocess
        gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,memory.used', '-u', '--format=csv,noheader'], 
                                  capture_output=True, text=True)
        if gpu_info.returncode == 0:
            for line in gpu_info.stdout.strip().split('\n'):
                print(f"  {line}")
        else:
            print("  GPU memory info not available")
    except:
        print("  nvidia-smi not found (might not be available in this environment)")

print("="*80)

## 9. Benchmark Summary & Recommendations

### Performance Insights

When GPU is available and properly configured:
- **Small operations**: CPU may be competitive due to lower overhead
- **Large matrix ops**: GPU typically shows 5-20× speedup  
- **Neural network ops**: GPU shows consistent improvements with larger batches
- **Element-wise ops**: GPU acceleration depends on operation complexity

### Recommendations for AxonScope

1. **Single Axon Simulations**: CPU is sufficient and memory-efficient
2. **Small Pools (< 100 axons)**: Choose based on available memory
3. **Large Pools (> 1000 axons)**: GPU significantly reduces execution time
4. **Development/Testing**: CPU recommended for fast iteration
5. **Production Runs**: GPU recommended for performance-critical workloads

### GPU Acceleration Strategy

- **Data Transfer**: Transfer data once, reuse on GPU
- **Batch Operations**: Amortize GPU overhead over multiple simulations
- **Mixed Precision**: Consider lower precision for faster computation
- **Pooling Strategy**: Use `dispatcher/` auto-batching for optimal grouping

### AxonScope Specific Notes

- **Dispatch Automatic**: Pool auto-grouping optimizes memory layout
- **Solver Selection**: Single-cable simpler than double-cable (Vi/Vperi/Ve)
- **JIT Compilation**: First run slower; subsequent runs benefit from caching
- **Memory Efficiency**: Frozen dataclasses minimize memory overhead

---

**Generated with JAX** | **AxonScope v0.1.0** | **Benchmark Date**: 2026-06-13

## 10. Advanced: AxonScope Specific Benchmark

Optional: Run actual AxonScope simulations to see real performance gains

In [ ]:
print("\n" + "="*80)
print("🧠 AXONSCOPE SIMULATION BENCHMARK")
print("="*80)

try:
    from axonscope import Q_
    import matplotlib.pyplot as plt
    
    # Simple HH axon setup
    print("\n📋 Setting up test simulations...")
    
    # Create a small pool of HH axons for benchmarking
    def create_hh_axons(n_axons: int = 10):
        """Create a pool of simple HH axons."""
        axons = []
        for i in range(n_axons):
            axon = HodgkinHuxley(
                diameter=Q_(1.0, "micrometer"),
                length=Q_(1000.0, "micrometer"),
                Ra=Q_(100.0, "ohm*cm"),
                Cm=Q_(1.0, "uF/cm2"),
                gNa=Q_(0.12, "S/cm2"),
                gK=Q_(0.036, "S/cm2"),
                gL=Q_(0.0003, "S/cm2"),
            )
            axons.append(AxonSimulation(axon))
        return axons
    
    # Create stimulus
    stimulus = Stimulus.pulse(
        amplitude=Q_(1.0, "microampere/cm2"),
        duration=Q_(0.1, "millisecond"),
        delay=Q_(0.05, "millisecond"),
    )
    
    # Recording setup
    recording = Recording(variables=["Vm"])
    
    print("✅ Setup complete")
    print("\n📊 Benchmarking simulations (5 axons, 10ms simulations)...")
    
    axons = create_hh_axons(5)
    
    # Run on CPU
    print("\n  🖥️  CPU Simulation...")
    cpu_axon_times = []
    for _ in range(3):
        start = time.time()
        results = CrankNicholson.solve(
            axons,
            stimulus=stimulus,
            recording=recording,
            t_stop=Q_(10.0, "millisecond"),
        )
        cpu_axon_times.append((time.time() - start) * 1000)
    
    cpu_axon_mean = np.mean(cpu_axon_times)
    print(f"     ⏱️  Mean time: {cpu_axon_mean:.2f} ms")
    
    # Run on GPU if available
    if gpu_available:
        print("\n  🚀 GPU Simulation...")
        gpu_axon_times = []
        for _ in range(3):
            start = time.time()
            results = CrankNicholson.solve(
                axons,
                stimulus=stimulus,
                recording=recording,
                t_stop=Q_(10.0, "millisecond"),
            )
            gpu_axon_times.append((time.time() - start) * 1000)
        
        gpu_axon_mean = np.mean(gpu_axon_times)
        print(f"     ⏱️  Mean time: {gpu_axon_mean:.2f} ms")
        
        speedup = cpu_axon_mean / gpu_axon_mean
        print(f"\n  📈 GPU Speedup: {speedup:.2f}×")
    
    print("\n✅ AxonScope benchmark complete!")
    
except Exception as e:
    print(f"⚠️  Note: Could not run AxonScope benchmarks: {str(e)}")
    print("    This is expected if AxonScope dependencies aren't fully installed.")
    print("    The generic JAX benchmarks above still provide useful CPU vs GPU comparison.")

print("="*80)

## 11. Resources & Documentation

### AxonScope References
- **Repository**: https://github.com/louisreg/AxonScope
- **Architecture Docs**: `docs/` folder in repository
- **Examples**: `examples/basic/` for didactic workflows
- **Benchmarks**: `benchmark/runtime/` for detailed profiling

### JAX Documentation
- **JAX Docs**: https://jax.readthedocs.io/
- **GPU Setup**: https://jax.readthedocs.io/en/latest/installation.html
- **Performance Tips**: https://jax.readthedocs.io/en/latest/notebooks/thinking_in_jax.html

### Google Colab Tips

**To enable GPU**:
1. Click `Runtime` menu
2. Select `Change runtime type`
3. Choose `GPU` (T4 or A100)
4. Click `Save`

**To check GPU status**:
```python
import jax
print(jax.devices())
```

**Memory limits**: 
- Colab CPU: ~12GB RAM
- Colab GPU: ~16GB GPU VRAM (T4) or ~40GB (A100)
- Session timeout: 12 hours

### Performance Optimization Checklist

- [ ] Enable GPU in Colab runtime
- [ ] Use JAX JIT compilation (`@jit` decorator)
- [ ] Batch multiple simulations together
- [ ] Pre-allocate arrays when possible
- [ ] Use `effects_barrier()` to ensure GPU completion
- [ ] Profile with `benchmark/` suite locally
- [ ] Monitor memory with `psutil` or GPU tools

### Troubleshooting

| Issue | Solution |
|-------|----------|
| GPU not detected | Ensure Runtime type is set to GPU in Colab |
| Out of memory | Reduce batch size or use smaller simulations |
| Slow first run | JAX JIT compiles on first execution |
| Installation errors | Try `pip install -e .` from repo root |

---

**Happy Benchmarking! 🚀** | Generated: 2026-06-13